<a href="https://colab.research.google.com/github/Heng1222/Ohsumed_classification/blob/feat-HeSH-WUPsimilarity/Model/task1_pytorch_framework(old%20version).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install torchinfo

In [ ]:
import torch
import time
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaModel, RobertaTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from peft import PeftModel, PeftConfig
from torchinfo import summary

# 1. 定義分類模型
class RobertaClassifier(nn.Module):
  def __init__(self, model_path_or_name, num_labels=23, freeze_backbone=True, useLoRA = False):
    super(RobertaClassifier, self).__init__()
    # 載入 RoBERTa
    self.roberta = RobertaModel.from_pretrained(model_path_or_name)
    # LoRA 掛載
    if(useLoRA):
      # config = PeftConfig.from_pretrained("ybelkada/opt-350m-lora")
      LoRA_folder = "maxbeettww/roberta-MeSH-lora-without-duplicate-path"
      self.roberta = PeftModel.from_pretrained(self.roberta, LoRA_folder)
    # 凍結 RoBERTa 參數
    if freeze_backbone:
      for param in self.roberta.parameters():
        param.requires_grad = False

    # 定義分類層 (NN Head)
    # RoBERTa-base 的 hidden_size 是 768
    self.classifier = nn.Sequential(
      nn.Linear(768, 1024),
      nn.LayerNorm(1024),
      nn.ReLU(),
      nn.Dropout(0.1),
      nn.Linear(1024, 512),
      nn.LayerNorm(512),
      nn.ReLU(),
      nn.Dropout(0.1),
      nn.Linear(512, 256),
      nn.LayerNorm(256),
      nn.ReLU(),
      nn.Dropout(0.1),
      nn.Linear(256, num_labels)
    )

    # summary model
    print("base model：\n\n",summary(self.roberta))
    print("classifer NN Head：\n\n", summary(self.classifier))


  def forward(self, input_ids, attention_mask):
    outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
    # 使用 CLS Token 的向量 (也可以換成 Mean Pooling)
    cls_output = outputs.last_hidden_state[:, 0, :]
    logits = self.classifier(cls_output)
    return logits

# 2. 資料集處理
class TextDataset(Dataset):
  def __init__(self, texts, labels, tokenizer, max_len=512):
    self.encodings = tokenizer(texts, truncation=True, truncation_side = 'left', padding='max_length', max_length=max_len, return_tensors="pt")
    self.labels = torch.tensor(labels, dtype=torch.long)

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, idx):
    return {
      'input_ids': self.encodings['input_ids'][idx],
      'attention_mask': self.encodings['attention_mask'][idx],
      'labels': self.labels[idx]
    }

# 3. 訓練與評估主程式
def run_experiment(model_name_or_path, train_df, test_df, num_epochs=30, useLoRA = False, folder_name = "default_model_name"):
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

  # 初始化
  tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
  model = RobertaClassifier(model_name_or_path, num_labels=23, useLoRA = useLoRA)

  model.to(device)

  # 測試集切割 9:1
  train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,       # 抽取 10% 作為測試集
    random_state=42,
    stratify=train_df['label']
  )
  train_loader = DataLoader(TextDataset(train_df['abstract'].tolist(), train_df['label'].tolist(), tokenizer), batch_size=32, shuffle=True)
  val_loader =  DataLoader(TextDataset(val_df['abstract'].tolist(), val_df['label'].tolist(), tokenizer), batch_size=32)
  test_loader = DataLoader(TextDataset(test_df['abstract'].tolist(), test_df['label'].tolist(), tokenizer), batch_size=32)

  optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=5e-4)
  criterion = nn.CrossEntropyLoss()

  # 紀錄 Loss 用於輸出圖表
  train_losses, val_losses = [], []
  best_val_loss = float('inf') # Track best validation loss
  best_model_state = None # To save the best model's state dict

  # 時間計時
  torch.cuda.synchronize()
  start = time.perf_counter()

  # 訓練迴圈
  for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
      optimizer.zero_grad()
      input_ids = batch['input_ids'].to(device)
      attention_mask = batch['attention_mask'].to(device)
      labels = batch['labels'].to(device)

      logits = model(input_ids, attention_mask)
      loss = criterion(logits, labels)
      loss.backward()
      optimizer.step()
      total_loss += loss.item()

    # Validation
    model.eval()
    val_total_loss = 0
    with torch.no_grad():
      for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} Validation"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        logits = model(input_ids, attention_mask)
        val_loss = criterion(logits, labels)
        val_total_loss += val_loss.item()

    avg_train = total_loss/len(train_loader)
    avg_val = val_total_loss/len(val_loader)
    train_losses.append(avg_train)
    val_losses.append(avg_val)
    print(f"Epoch {epoch+1} Training Loss: {avg_train:.4f} Validation Loss: {avg_val:.4f}")

    # Save the best model based on validation loss
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        best_model_state = model.state_dict()
        print(f"  --> Best model saved with validation loss: {best_val_loss:.4f}")

  # 時間計時終止
  torch.cuda.synchronize()
  end = time.perf_counter()

  # Load the best model before testing
  if best_model_state is not None:
      model.load_state_dict(best_model_state)
      print("Loaded best model for final evaluation.")

  # test
  model.eval()
  all_preds = []
  all_labels = []
  with torch.no_grad():
    for batch in test_loader:
      input_ids = batch['input_ids'].to(device)
      attention_mask = batch['attention_mask'].to(device)
      labels = batch['labels'].to(device)

      logits = model(input_ids, attention_mask)
      preds = torch.argmax(logits, dim=1)
      all_preds.extend(preds.cpu().numpy())
      all_labels.extend(labels.cpu().numpy())

  # 訓練時間
  print(f"訓練時間: {end - start:.2f} 秒")
  # 詳細報表
  print("\n--- Final Evaluation Report ---")
  print(classification_report(all_labels, all_preds))
  # Loss 趨勢圖
  plt.figure(figsize=(10, 6))
  plt.plot(train_losses, label='Train Loss')
  plt.plot(val_losses, label='Val Loss')
  plt.title('Training and Validation Semantic Loss')
  plt.xlabel('Epochs')
  plt.ylabel('Loss')
  plt.legend()
  plt.show()
  # # 儲存模型
  # model.save_pretrained(folder_name)
  # print(f"model saved to '{folder_name}'.")
  return all_labels, all_preds

# 執行範例
if __name__ == "__main__":
  # 資料讀取
  url = 'ohsumed_dataset.csv'
  full_df = pd.read_csv(url)
  columns = ['label', 'abstract']
  full_df = full_df[columns]
  # 將分類標籤從字串轉換為數字
  unique_labels = full_df['label'].unique()
  label_to_id = {label: i for i, label in enumerate(sorted(unique_labels))}
  # 將 'label' 欄位轉換為數字
  full_df['label'] = full_df['label'].map(label_to_id)
  display(full_df.head())
  # 找出最少樣本的類別有多少筆
  min_sample_size = full_df['label'].value_counts().min()
  print(f"每個類別將統一採樣至: {min_sample_size} 筆")

  # 每個類別隨機抽取相同筆數
  full_df = full_df.groupby('label').apply(
      lambda x: x.sample(n=min_sample_size, random_state=42)
  ).reset_index(drop=True)
  # 分層抽樣
  # full_df = full_df.groupby('label', group_keys=False).apply(
  #   lambda x: x.sample(frac=0.1, random_state=42)
  # )
  train_df, test_df = train_test_split(
    full_df,
    test_size=0.1,       # 抽取 10% 作為測試集
    random_state=42,
    stratify=full_df['label']
  )
  print("train data size: ", train_df.shape)
  print("test data size: ", test_df.shape)

  # 實驗 1: 使用 原版RoBERTa
  # print("Running Experiment with Base RoBERTa...")
  # run_experiment('roberta-base', train_df, test_df, folder_name = "RoBERTa_based")

  # 實驗 2: 使用 MLM
  # print("Running Experiment with Custom RoBERTa...")
  # run_experiment('maxbeettww/roberta-ohsumed-mlm', train_df, test_df, folder_name = "RoBERTa_MLM")

  # 實驗 3: 使用 LoRA
  print("Running Experiment with RoBERTa + LoRA...")
  run_experiment('roberta-base', train_df, test_df, useLoRA = True, folder_name = "RoBERTa_LoRA")

,label,abstract
0,0,A retrospective evaluation of Haemophilus infl...
1,0,Many different materials are available for aug...
2,0,The purpose of this article is to alert clinic...
3,0,Septic arthritis developed in a neonate after ...
4,0,Many methods have been devised to prevent asce...


每個類別將統一採樣至: 427 筆
train data size:  (8838, 2)
test data size:  (983, 2)
Running Experiment with RoBERTa + LoRA...


/tmp/ipykernel_9192/932750950.py:211: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  full_df = full_df.groupby('label').apply(


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


base model：

Layer (type:depth-idx)                                                 Param #
PeftModel                                                              --
├─LoraModel: 1-1                                                       --
│    └─RobertaModel: 2-1                                               --
│    │    └─RobertaEmbeddings: 3-1                                     (39,000,576)
│    │    └─RobertaEncoder: 3-2                                        (86,971,392)
│    │    └─RobertaPooler: 3-3                                         (590,592)
Total params: 126,562,560
Trainable params: 0
Non-trainable params: 126,562,560
classifer NN Head：

Layer (type:depth-idx)                   Param #
Sequential                               --
├─Linear: 1-1                            787,456
├─LayerNorm: 1-2                         2,048
├─ReLU: 1-3                              --
├─Dropout: 1-4                           --
├─Linear: 1-5                            524,800
├─LayerNorm

Epoch 1:  14%|█▎        | 34/249 [00:45<04:19,  1.21s/it]